# Joint QCD mass histograms (cross-section weighted)

Builds joint, stacked histograms over all QCD HT slices in `datasets.json`. Each
slice is weighted by `cross_section / N_original`, where `N_original` is the
original number of events read by the slimmer, taken from the **first bin of the
`cutflow`** histogram and summed over the slice's files.

Prereqs:
1. `python scripts/make_dataset_json.py <eos_path> -o datasets.json`
2. Cross sections are read from `run3-mj-pass-the-aux/mj_samples_xs.json`
   (point `XS_JSON` below at your checkout of that repo).

In [ ]:
import sys, pathlib, json

# Make the package importable without `pip install -e .` (src/ layout).
sys.path.insert(0, str(pathlib.Path.cwd().parent / "src"))

import numpy as np
import awkward as ak
import uproot
import hist
import vector
import matplotlib.pyplot as plt
from coffea.nanoevents import NanoEventsFactory, BaseSchema

from run3_mj_analyzer.fileset import load_fileset

vector.register_awkward()  # enables Momentum4D behaviors (.mass, .px, ...)

In [ ]:
JSON_PATH = "../datasets.json"  # written by scripts/make_dataset_json.py

fileset = load_fileset(JSON_PATH)
print(f"{len(fileset)} slices:")
for name, ds in fileset.items():
    print(f"  {name}: {len(ds['files'])} files")

## Cross sections

Loaded from the shared aux repo `run3-mj-pass-the-aux/mj_samples_xs.json`
(`{dataset: {"xs_pb": ..., "sig_pb": ...}}`); we use `xs_pb`. The check below
raises if any slice in the fileset is missing from that file.

In [ ]:
# Path to the cross-section JSON in the run3-mj-pass-the-aux checkout.
# Default assumes it sits next to run3-mj-analyzer; adjust if yours differs.
XS_JSON = "../../run3-mj-pass-the-aux/mj_samples_xs.json"

with open(XS_JSON) as f:
    _xs = json.load(f)
CROSS_SECTIONS = {name: info["xs_pb"] for name, info in _xs.items()}

missing = [d for d in fileset if d not in CROSS_SECTIONS]
if missing:
    raise ValueError(
        f"No cross section in {XS_JSON} for these slices:\n  "
        + "\n  ".join(missing)
    )
print(f"Loaded {len(CROSS_SECTIONS)} cross sections from {XS_JSON}")

## Helpers and per-slice weight

`weight = cross_section / N_original` (optionally scaled by an integrated
luminosity). `N_original` is summed over the slice's files from the first
`cutflow` bin.

In [ ]:
LUMI = 1.0  # pb^-1. Leave 1.0 for pure xsec/N weights; set to lumi for yields.


def jet_p4(events):
    """Per-event Momentum4D jets from the slimmer's ScoutingPFJet record."""
    if "ScoutingPFJet" in events.fields:
        j = events["ScoutingPFJet"]
        pt, eta, phi, m = j.pt, j.eta, j.phi, j.m
    else:  # flat fallback
        pt = events["ScoutingPFJet_pt"]
        eta = events["ScoutingPFJet_eta"]
        phi = events["ScoutingPFJet_phi"]
        m = events["ScoutingPFJet_m"]
    return ak.zip(
        {"pt": pt, "eta": eta, "phi": phi, "mass": m}, with_name="Momentum4D"
    )


def original_event_count(filepath):
    """Original events read for a file = first bin of its 'cutflow' histogram."""
    with uproot.open(filepath) as f:
        cutflow = f["cutflow"]
        try:
            return float(cutflow.values()[0])
        except Exception:
            return float(cutflow.to_hist().values()[0])


def slice_n_original(filepaths):
    """Sum the first cutflow bin over every file in a slice (cheap: no tree read)."""
    return sum(original_event_count(fp) for fp in filepaths)

## Fill the joint histograms (single pass over all slices)

Each histogram has a `dataset` category axis (for stacking) and a mass axis,
with `Weight` storage so statistical errors propagate.

In [ ]:
DATASETS = list(fileset)
N_FILES = None  # set an int to limit files/slice while testing (under-fills,
                # but the weight still uses the FULL slice N_original)

cat = hist.axis.StrCategory(DATASETS, name="dataset", label="QCD slice")
h_lead = hist.Hist(
    cat, hist.axis.Regular(60, 0, 120, name="m", label="Leading-jet mass [GeV]"),
    storage=hist.storage.Weight(),
)
h_dijet = hist.Hist(
    cat, hist.axis.Regular(60, 0, 2000, name="m", label="Leading dijet mass [GeV]"),
    storage=hist.storage.Weight(),
)
h_sys = hist.Hist(
    cat, hist.axis.Regular(60, 0, 5000, name="m", label="All-jet system mass [GeV]"),
    storage=hist.storage.Weight(),
)

for name in DATASETS:
    # N_original uses ALL files of the slice (correct normalization). Reading a
    # cutflow is cheap -- it does not touch the events tree.
    all_paths = list(fileset[name]["files"])
    n_orig = slice_n_original(all_paths)
    weight = CROSS_SECTIONS[name] / n_orig * LUMI
    print(f"{name}: N_orig={n_orig:,.0f}  xsec={CROSS_SECTIONS[name]:g} pb  w={weight:.3e}")

    items = list(fileset[name]["files"].items())
    if N_FILES:
        items = items[:N_FILES]
    for fp, tree in items:
        events = NanoEventsFactory.from_root(
            {fp: tree}, schemaclass=BaseSchema, delayed=False
        ).events()
        jets = jet_p4(events)
        njet = ak.num(jets)

        lead = jets[njet >= 1][:, 0]
        h_lead.fill(dataset=name, m=ak.to_numpy(lead.mass), weight=weight)

        two = jets[njet >= 2]
        dijet = two[:, 0] + two[:, 1]
        h_dijet.fill(dataset=name, m=ak.to_numpy(dijet.mass), weight=weight)

        system = ak.zip(
            {
                "px": ak.sum(jets.px, axis=1),
                "py": ak.sum(jets.py, axis=1),
                "pz": ak.sum(jets.pz, axis=1),
                "E": ak.sum(jets.energy, axis=1),
            },
            with_name="Momentum4D",
        )
        h_sys.fill(dataset=name, m=ak.to_numpy(system.mass), weight=weight)

print("done filling")

## Plots

Stacked per-slice contributions (filled) plus the joint QCD total (black step).
With `LUMI = 1.0` the y axis is differential cross section (pb per bin).

In [ ]:
def plot_joint(h, title, logy=True):
    fig, ax = plt.subplots(figsize=(7, 5))
    h.stack("dataset").plot(stack=True, histtype="fill", ax=ax)
    h.project("m").plot(histtype="step", color="black", linewidth=1.5,
                         label="QCD total", ax=ax)
    ax.set_title(title)
    ax.set_ylabel("cross section [pb] / bin" if LUMI == 1.0 else "events")
    if logy:
        ax.set_yscale("log")
    ax.legend(fontsize=6, ncol=2)
    plt.show()


plot_joint(h_lead, "Leading-jet mass (QCD, xsec-weighted)")

In [ ]:
plot_joint(h_dijet, "Leading dijet mass (QCD, xsec-weighted)")

In [ ]:
plot_joint(h_sys, "All-jet system mass (QCD, xsec-weighted)")